***Transformer Version of the chatbot***

In [21]:
!pip install sentence-transformers faiss-cpu transformers accelerate bitsandbytes nltk bert-score

import os
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import T5ForConditionalGeneration, T5Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import nltk
nltk.download('punkt')

import time
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score
import json

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [22]:
corpus_dbs = []
folder_path_training = "/content/Research-Chatbot"

for file_name in os.listdir(folder_path_training):
    file_path = os.path.join(folder_path_training, file_name)
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            corpus_dbs.extend([para.strip() for para in content.split('\n') if len(para.strip()) > 50])
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print(f"Loaded {len(corpus_dbs)} paragraphs.")


Loaded 3223 paragraphs.


In [23]:
model_dbs_transformer = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings_dbs = model_dbs_transformer.encode(corpus_dbs, show_progress_bar=True, convert_to_numpy=True)


Batches:   0%|          | 0/101 [00:00<?, ?it/s]

In [24]:
embedding_dim_transformer = corpus_embeddings_dbs.shape[1]  # Should be 384 for all-MiniLM-L6-v2
index_dbs = faiss.IndexFlatL2(embedding_dim_transformer)
index_dbs.add(corpus_embeddings_dbs)

print(f"FAISS index_dbs built with {index_dbs.ntotal} vectors.")


FAISS index_dbs built with 3223 vectors.


In [25]:
def retrieve_passages(query, top_k=3):
    query_embedding = model_dbs_transformer.encode([query], convert_to_numpy=True)
    distances, indices = index_dbs.search(query_embedding, top_k)
    results = [corpus_dbs[idx] for idx in indices[0]]
    return results

In [26]:
t5_model_dbs = T5ForConditionalGeneration.from_pretrained("t5-base")
t5_tokenizer_dbs = T5Tokenizer.from_pretrained("t5-base")

def generate_answer(query, context_passages):
    context = " ".join(context_passages)
    prompt = f"question: {query} context: {context}"
    inputs = t5_tokenizer_dbs(prompt, return_tensors="pt", truncation=True, padding=True)
    outputs = t5_model_dbs.generate(**inputs, max_length=128)
    answer = t5_tokenizer_dbs.decode(outputs[0], skip_special_tokens=True)
    return answer


In [27]:
query = "What are library opening hours?"
passages = retrieve_passages(query)

print("\nTop Retrieved Passages:")
for i, p in enumerate(passages):
    print(f"[{i+1}] {p}\n")

answer = generate_answer(query, passages)
print("Generated Answer:\n", answer)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



Top Retrieved Passages:
[1] What hours is the library open?                                                                                                                            10

[2] As the days grow longerour opening hours get shorter. We are now open Monday to Saturday9am to 5pm.See all the opening hours here!

[3] As the days grow longerour opening hours get shorter. We are now open Monday to Saturday9am to 5pm.See all the opening hours here!

Generated Answer:
 Monday to Saturday9am to 5pm


In [28]:
def start_chatbot(top_k=3):
    print("DBS Chatbot is ready Type 'exit' to quit.\n")

    while True:
        query = input("You: ")
        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        retrieved_passages = retrieve_passages(query)
        answer = generate_answer(query,retrieved_passages)

        print("DBS Chatbot:", answer)
        print("-" * 60)

In [29]:
start_chatbot()


DBS Chatbot is ready Type 'exit' to quit.

You: student population of DBS?
DBS Chatbot: over 9,000
------------------------------------------------------------
You: exit
Goodbye!


In [30]:
# Load the TinyLLaMA Chat model
model_id_tinyllm_dbs = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_tinyllm_dbs = AutoTokenizer.from_pretrained(model_id_tinyllm_dbs)
model_tinyllm_dbs = AutoModelForCausalLM.from_pretrained(model_id_tinyllm_dbs, device_map="auto", torch_dtype="auto")

In [31]:
def refine_answer_llama(query, raw_answer):
    prompt = f"""<|system|>You are an intelligent chatbot. Your role is to convert the retreived chunks into humanized text..<|end|>
<|user|>Question: {query}
Answer: {raw_answer}
Convert the following text to a chatbot response. Add greetings and ask the user if they have any further questions. AT the end of the conversion, say Thank You.<|end|>
<|assistant|>"""

    inputs = tokenizer_tinyllm_dbs(prompt, return_tensors="pt").to(model_tinyllm_dbs.device)
    outputs = model_tinyllm_dbs.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer_tinyllm_dbs.eos_token_id
    )

    full_output = tokenizer_tinyllm_dbs.decode(outputs[0], skip_special_tokens=True)
    # Extract the assistant's response
    if "<|assistant|>" in full_output:
        return full_output.split("<|assistant|>")[-1].strip()
    return full_output.strip()


In [32]:
query = "How many books does the DBS Library have?"
raw_answer = "over 43,000"

#refined = refine_answer_llama(query, raw_answer)
#print("📘 Refined Answer:", refined)

In [33]:
def start_chatbot_v2(top_k=3):
    print("DBS Chatbot is ready! Type 'exit' to quit.\n")

    while True:
        query = input("You: ")
        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        retrieved_passages = retrieve_passages(query)
        answer = generate_answer(query,retrieved_passages)
        print("DBS Chatbot original answer:", answer)
        print("-" * 60)

        refined_answer = refine_answer_llama(query, answer)

        # Step 5: Show the answer
        print("DBS Chatbot refined answer:", refined_answer)
        print("-" * 60)

In [34]:
#start_chatbot_v2()

Evaluation Pipeline

In [35]:
test_set = [
    {"query": "How many books DBS Library has?", "expected_answer": "over 43,000"},
     {"query": "Where can I access DBS WiFi?", "expected_answer": ""},
      {"query": "What are library opening hours?", "expected_answer": " 24 hours a day"},
       {"query": "What ratings did DBS earned?", "expected_answer": " 4 stars"},
        {"query": "how to view Library account?", "expected_answer": ""},
         {"query": "Are the Guides to Library resources for students with disabilities are also available in the Library?", "expected_answer": " on the library website"},
          {"query": "How many university partnerships does DBS has developed?", "expected_answer": " over 75"}
  ]

  #create a program to load this from file





In [36]:
file_path_test = r"/content/Research-Chatbot/Transformer_Test_DataSet.json"


# Load test_set from file
def load_test_set(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"The file {path} does not exist.")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

if __name__ == "__main__":
    try:
        test_set = load_test_set(file_path_test)
        print("Test set loaded successfully.")
        for item in test_set:
            print(f"Query: {item.get('query', '')}")
            print(f"Expected Answer: {item.get('expected_answer', '')}")
            print("-" * 50)
    except Exception as e:
        print(f"Error: {e}")


Test set loaded successfully.
Query: What is PSI?
Expected Answer: professional body for psychology in Ireland
--------------------------------------------------
Query: What country has the highest reputation for leading independent colleges?
Expected Answer: Ireland
--------------------------------------------------
Query: student population of DBS?
Expected Answer: over 9,000
--------------------------------------------------
Query: What is the name of Ireland's largest independent third-level college?
Expected Answer: Dublin Business School
--------------------------------------------------
Query: When was Dublin Business School established?
Expected Answer: 1975
--------------------------------------------------
Query: When was DBS established?
Expected Answer: 1975
--------------------------------------------------
Query: DBS was established in which year?
Expected Answer: 1975
--------------------------------------------------
Query: What is the name of the society that represent

In [37]:
#test_set

In [38]:
smoothie = SmoothingFunction().method4

def evaluate_transformer_model(test_set):
    results = []
    total_time = 0
    all_generated = []
    all_expected = []

    for i, item in enumerate(test_set):
        query = item["query"]
        expected = item["expected_answer"]
        print(f"Evaluating Query {i+1}/{len(test_set)}")
        start_time = time.time()
        retrieved_passages = retrieve_passages(query)
        generated = generate_answer(query,retrieved_passages)
        end_time = time.time()

        response_time = end_time - start_time
        total_time += response_time

        # Save for BERTScore later
        all_generated.append(generated)
        all_expected.append(expected)

        # Exact match
        exact_match = int(expected.lower() in generated.lower())

        # BLEU Score
        reference = [expected.split()]
        candidate = generated.split()
        bleu = sentence_bleu(reference, candidate, smoothing_function=smoothie)

        results.append({
            "Query": query,
            "Generated": generated,
            "Expected": expected,
            "ExactMatch": exact_match,
            "BLEU": bleu,
            "TimeTaken": response_time
        })

    # BERTScore
    P, R, F1 = bert_score(all_generated, all_expected, lang="en", verbose=True)
    avg_bertscore_f1 = F1.mean().item()

    # Summary metrics
    accuracy = sum(r["ExactMatch"] for r in results) / len(results)
    avg_bleu = sum(r["BLEU"] for r in results) / len(results)
    avg_time = total_time / len(results)

    print(f"\n--- Evaluation Summary ---")
    print(f"Accuracy (Exact Match): {accuracy:.2f}")
    print(f"Average BLEU Score: {avg_bleu:.2f}")
    print(f"Average BERTScore F1: {avg_bertscore_f1:.2f}")
    print(f"Average Inference Time: {avg_time:.2f} seconds\n")

    return results


In [ ]:
results = evaluate_transformer_model(test_set)
results

Evaluating Query 1/172
Evaluating Query 2/172
Evaluating Query 3/172
Evaluating Query 4/172
Evaluating Query 5/172
Evaluating Query 6/172
Evaluating Query 7/172
Evaluating Query 8/172
Evaluating Query 9/172
Evaluating Query 10/172
Evaluating Query 11/172
Evaluating Query 12/172
Evaluating Query 13/172
Evaluating Query 14/172
Evaluating Query 15/172
Evaluating Query 16/172
Evaluating Query 17/172
Evaluating Query 18/172
Evaluating Query 19/172
Evaluating Query 20/172
Evaluating Query 21/172
Evaluating Query 22/172
Evaluating Query 23/172
Evaluating Query 24/172
Evaluating Query 25/172
Evaluating Query 26/172
Evaluating Query 27/172
Evaluating Query 28/172
Evaluating Query 29/172
Evaluating Query 30/172
Evaluating Query 31/172
Evaluating Query 32/172
Evaluating Query 33/172
Evaluating Query 34/172
Evaluating Query 35/172
Evaluating Query 36/172
Evaluating Query 37/172
Evaluating Query 38/172
Evaluating Query 39/172
